# 07 - Results Interpretation

**Purpose:** consolidate the final results around the project question:

> Can time-series history, social-network exposure, review-language signals, and stronger scikit-learn model families help forecast short-term shifts in community attention toward local Yelp businesses?

This interpretation uses the corrected forecasting cohort: **876 businesses** and **68,249 business-month rows** from the 2015-2021 modeling window, excluding rows before each business's first observed modeling-window review.

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
PULSE_METRICS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_metrics.csv"
PULSE_PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_predictions.csv"
PULSE_TOPK_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_topk_metrics.csv"
PULSE_CALIBRATION_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_calibration.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
pulse_metrics = pd.read_csv(PULSE_METRICS_OUTPUT_PATH)
pulse_predictions = pd.read_csv(PULSE_PREDICTIONS_OUTPUT_PATH)
pulse_topk_metrics = pd.read_csv(PULSE_TOPK_OUTPUT_PATH)
pulse_calibration = pd.read_csv(PULSE_CALIBRATION_OUTPUT_PATH)
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,task,model,train_period,validation_period,test_period,rows,MAE,RMSE,WAPE,model_family,feature_count
0,primary_covid_test,review_count_regression,Baseline: last month,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.601170,3.007150,0.664292,temporal_baseline,0
1,primary_covid_test,review_count_regression,Baseline: rolling 3-month avg,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.646071,3.364436,0.682921,temporal_baseline,0
2,primary_covid_test,review_count_regression,ML: historical + SNA,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.970705,3.525411,0.817604,RandomForestRegressor,33
3,primary_covid_test,review_count_regression,ML: historical + NLP,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.996907,3.570527,0.828475,RandomForestRegressor,18
4,primary_covid_test,review_count_regression,ML: historical + business,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.997847,3.636709,0.828865,RandomForestRegressor,16
5,primary_covid_test,review_count_regression,ML: historical,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.004549,3.607005,0.831646,RandomForestRegressor,10
6,primary_covid_test,review_count_regression,ML: all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.034262,3.608341,0.843973,RandomForestRegressor,47
7,primary_covid_test,review_count_regression,ML: HGB selected top 20,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.114068,3.483930,0.877083,SelectKBest + HistGradientBoostingRegressor,47
8,primary_covid_test,review_count_regression,ML: HGB all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.179276,3.982742,0.904136,HistGradientBoostingRegressor,47
9,primary_covid_test,review_count_regression,ML: Poisson all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.615199,4.164493,1.084991,PoissonRegressor,47


## Regression Interpretation

Summarize which model family performs best in each time split and whether stronger alternatives improve on the simple temporal baselines. This pass compares Random Forests, HistGradientBoosting, Poisson count models, and selected-feature variants.

In [2]:
regression_summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    hgb_all = split_metrics[split_metrics["model"] == "ML: HGB all modalities"].iloc[0]
    poisson_all = split_metrics[split_metrics["model"] == "ML: Poisson all modalities"].iloc[0]
    selected_hgb = split_metrics[split_metrics["model"] == "ML: HGB selected top 20"].iloc[0]
    regression_summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_family": best["model_family"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "business_WAPE": business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "nlp_WAPE": nlp["WAPE"],
        "all_modalities_WAPE": all_modalities["WAPE"],
        "hgb_all_WAPE": hgb_all["WAPE"],
        "poisson_all_WAPE": poisson_all["WAPE"],
        "selected_hgb_WAPE": selected_hgb["WAPE"],
        "best_vs_last_month_relative_change": (best["WAPE"] - split_metrics[split_metrics["model"] == "Baseline: last month"].iloc[0]["WAPE"]) / split_metrics[split_metrics["model"] == "Baseline: last month"].iloc[0]["WAPE"],
        "all_vs_historical_relative_change": (all_modalities["WAPE"] - hist["WAPE"]) / hist["WAPE"],
        "all_vs_business_relative_change": (all_modalities["WAPE"] - business["WAPE"]) / business["WAPE"],
    })
regression_summary = pd.DataFrame(regression_summary_rows)
regression_summary

,split,best_model,best_family,best_WAPE,historical_WAPE,business_WAPE,sna_WAPE,nlp_WAPE,all_modalities_WAPE,hgb_all_WAPE,poisson_all_WAPE,selected_hgb_WAPE,best_vs_last_month_relative_change,all_vs_historical_relative_change,all_vs_business_relative_change
0,primary_covid_test,Baseline: last month,temporal_baseline,0.664292,0.831646,0.828865,0.817604,0.828475,0.843973,0.904136,1.084991,0.877083,0.000000,0.014823,0.018227
1,secondary_pre_covid_test,ML: HGB all modalities,HistGradientBoostingRegressor,0.380808,0.388882,0.381904,0.388126,0.385845,0.381137,0.380808,0.505761,0.388816,-0.148635,-0.019915,-0.002010


## Pulse Interpretation

Summarize pulse-classification performance with class balance and probability quality in mind:

- F1 uses a cutoff tuned on the validation period;
- PR-AUC and top-k metrics assess ranking quality for rare pulse events;
- Brier score and calibration bins assess whether pulse probabilities behave like useful risk estimates.

In [3]:
# Pulse summaries emphasize rare-event detection and probability quality rather than overall accuracy.
pulse_summary_rows = []
for split_name, split_metrics in pulse_metrics.groupby("split"):
    ranked = split_metrics.sort_values(["F1", "PR_AUC"], ascending=[False, False]).reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    hgb_all = split_metrics[split_metrics["model"] == "ML: HGB all modalities"].iloc[0]
    logistic_all = split_metrics[split_metrics["model"] == "ML: Logistic all modalities"].iloc[0]
    selected_hgb = split_metrics[split_metrics["model"] == "ML: HGB selected top 20"].iloc[0]
    best_brier = split_metrics.sort_values("Brier").iloc[0]

    split_topk_10 = pulse_topk_metrics[
        (pulse_topk_metrics["split"] == split_name)
        & (pulse_topk_metrics["k_fraction"] == 0.10)
    ].copy()
    best_topk_10 = split_topk_10.sort_values(["precision_at_k", "recall_at_k"], ascending=[False, False]).iloc[0]
    all_topk_10 = split_topk_10[split_topk_10["model"] == "ML: all modalities"].iloc[0]

    pulse_summary_rows.append({
        "split": split_name,
        "positive_rate": all_modalities["positive_rate"],
        "best_model": best["model"],
        "best_family": best["model_family"],
        "best_F1": best["F1"],
        "best_PR_AUC": best["PR_AUC"],
        "best_Brier": best["Brier"],
        "best_threshold": best["decision_threshold"],
        "best_validation_F1": best["validation_F1"],
        "best_brier_model": best_brier["model"],
        "best_brier": best_brier["Brier"],
        "historical_F1": hist["F1"],
        "business_F1": business["F1"],
        "sna_F1": sna["F1"],
        "nlp_F1": nlp["F1"],
        "all_modalities_F1": all_modalities["F1"],
        "all_modalities_PR_AUC": all_modalities["PR_AUC"],
        "hgb_all_F1": hgb_all["F1"],
        "hgb_all_PR_AUC": hgb_all["PR_AUC"],
        "logistic_all_F1": logistic_all["F1"],
        "logistic_all_PR_AUC": logistic_all["PR_AUC"],
        "selected_hgb_F1": selected_hgb["F1"],
        "selected_hgb_PR_AUC": selected_hgb["PR_AUC"],
        "selected_hgb_Brier": selected_hgb["Brier"],
        "all_modalities_threshold": all_modalities["decision_threshold"],
        "best_precision_at_10pct_model": best_topk_10["model"],
        "best_precision_at_10pct": best_topk_10["precision_at_k"],
        "best_recall_at_10pct": best_topk_10["recall_at_k"],
        "all_modalities_precision_at_10pct": all_topk_10["precision_at_k"],
        "all_modalities_recall_at_10pct": all_topk_10["recall_at_k"],
    })
pulse_summary = pd.DataFrame(pulse_summary_rows)
pulse_summary

,split,positive_rate,best_model,best_family,best_F1,best_PR_AUC,best_Brier,best_threshold,best_validation_F1,best_brier_model,...,logistic_all_PR_AUC,selected_hgb_F1,selected_hgb_PR_AUC,selected_hgb_Brier,all_modalities_threshold,best_precision_at_10pct_model,best_precision_at_10pct,best_recall_at_10pct,all_modalities_precision_at_10pct,all_modalities_recall_at_10pct
0,primary_covid_test,0.102407,Baseline: rising recent activity,rule_baseline,0.280882,0.148129,0.200200,0.500,0.215558,ML: HGB selected top 20,...,0.236159,0.275767,0.230490,0.092019,0.400,ML: Logistic all modalities,0.277699,0.271249,0.225392,0.220158
1,secondary_pre_covid_test,0.131957,ML: HGB selected top 20,SelectKBest + HistGradientBoostingClassifier,0.308690,0.237247,0.110175,0.155,0.382133,ML: HGB selected top 20,...,0.201392,0.308690,0.237247,0.110175,0.425,ML: historical,0.301331,0.228551,0.283270,0.214852


In [4]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 63
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 876
row_count: 68249
feature_month_min: 2015-01
feature

In [5]:
for _, row in regression_summary.iterrows():
    split = row["split"]
    best_vs_last_month = row["best_vs_last_month_relative_change"] * 100
    all_vs_hist = row["all_vs_historical_relative_change"] * 100
    all_vs_business = row["all_vs_business_relative_change"] * 100
    print(f"{split} regression:")
    print(f"  Best model: {row['best_model']} ({row['best_family']}) with WAPE={row['best_WAPE']:.4f}")
    print(f"  Best vs last-month baseline: {best_vs_last_month:+.2f}%")
    print(f"  HGB all modalities WAPE: {row['hgb_all_WAPE']:.4f}")
    print(f"  Poisson all modalities WAPE: {row['poisson_all_WAPE']:.4f}")
    print(f"  Selected HGB WAPE: {row['selected_hgb_WAPE']:.4f}")
    print(f"  All modalities vs historical: {all_vs_hist:+.2f}%")
    print(f"  All modalities vs historical+business: {all_vs_business:+.2f}%")

print()
for _, row in pulse_summary.iterrows():
    split = row["split"]
    print(f"{split} attention pulses:")
    print(f"  Positive rate: {row['positive_rate']:.3f}")
    print(f"  Best thresholded model: {row['best_model']} ({row['best_family']}) with F1={row['best_F1']:.4f}, PR-AUC={row['best_PR_AUC']:.4f}, Brier={row['best_Brier']:.4f}")
    print(f"  Best model threshold: {row['best_threshold']:.3f}; validation F1={row['best_validation_F1']:.4f}")
    print(f"  Best Brier model: {row['best_brier_model']} with Brier={row['best_brier']:.4f}")
    print(f"  HGB selected top 20: F1={row['selected_hgb_F1']:.4f}, PR-AUC={row['selected_hgb_PR_AUC']:.4f}, Brier={row['selected_hgb_Brier']:.4f}")
    print(f"  Logistic all modalities: F1={row['logistic_all_F1']:.4f}, PR-AUC={row['logistic_all_PR_AUC']:.4f}")
    print(f"  Best precision@10%: {row['best_precision_at_10pct_model']} with precision={row['best_precision_at_10pct']:.4f}, recall={row['best_recall_at_10pct']:.4f}")

primary_covid_test regression:
  Best model: Baseline: last month (temporal_baseline) with WAPE=0.6643
  Best vs last-month baseline: +0.00%
  HGB all modalities WAPE: 0.9041
  Poisson all modalities WAPE: 1.0850
  Selected HGB WAPE: 0.8771
  All modalities vs historical: +1.48%
  All modalities vs historical+business: +1.82%
secondary_pre_covid_test regression:
  Best model: ML: HGB all modalities (HistGradientBoostingRegressor) with WAPE=0.3808
  Best vs last-month baseline: -14.86%
  HGB all modalities WAPE: 0.3808
  Poisson all modalities WAPE: 0.5058
  Selected HGB WAPE: 0.3888
  All modalities vs historical: -1.99%
  All modalities vs historical+business: -0.20%

primary_covid_test attention pulses:
  Positive rate: 0.102
  Best thresholded model: Baseline: rising recent activity (rule_baseline) with F1=0.2809, PR-AUC=0.1481, Brier=0.2002
  Best model threshold: 0.500; validation F1=0.2156
  Best Brier model: ML: HGB selected top 20 with Brier=0.0920
  HGB selected top 20: F1=0.2

## Interpretation Structure

1. **Time series:** recent review patterns remain the strongest reference point for raw review-count forecasting, especially in the COVID-era test.
2. **Model families:** HistGradientBoosting helps pulse detection and pre-COVID count forecasting, while Poisson regression is not competitive for this sparse, disrupted review-count task.
3. **Feature selection:** selected top-20 HGB models give the cleanest pulse results, suggesting that smaller all-modality subsets can be more useful than throwing every feature into one model.
4. **Probability quality:** Brier score and calibration curves are necessary because high ranking performance does not automatically mean reliable pulse probabilities.
5. **SNA/NLP:** social and text features remain valuable for the multimodal story, but their benefit appears through selected nonlinear models more than through simple full-feature ablations.
6. **Target design:** attention pulses are better evaluated as both classification and ranking. Top-k metrics are useful because analysts may only inspect the highest-risk business-months.

## Final Position

The improved modeling pass makes the final story stronger: the project now compares simple temporal baselines, Random Forests, gradient boosting, Poisson count regression, logistic classification, selected-feature models, top-k retrieval, and probability calibration.

The main empirical result is still grounded rather than exaggerated. Simple temporal baselines remain very hard to beat for COVID-era review-count forecasting. For attention pulses, however, HistGradientBoosting with selected features improves the ML story: it gives stronger F1, stronger PR-AUC, and better Brier scores than the earlier full Random Forest ablation. This supports the idea that community attention pulses are better treated as a rare-event risk-ranking problem than as a plain count forecast.

Key limitations:

- Yelp friendship links are static.
- SNA features measure exposure, not causal influence.
- NLP features are lightweight lexicon/text-length signals.
- COVID-era disruption changes predictability.
- Review activity is a proxy for Yelp attention, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.
- Calibration is diagnostic, not a guarantee that probabilities transfer outside this dataset.